In [2]:
# Task 2 — Thematic Analysis using TF-IDF
# --------------------------------------

# Install if needed:
# pip install pandas scikit-learn nltk

import pandas as pd
import nltk

from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter

# Download stopwords once
nltk.download('stopwords')

from nltk.corpus import stopwords

# --------------------------------------
# 1. Load dataset
# --------------------------------------
df = pd.read_csv("reviews_with_sentiment.csv")

# Remove missing reviews
df = df.dropna(subset=["review text"])

# --------------------------------------
# 2. TF-IDF Vectorizer
# --------------------------------------
stop_words = stopwords.words("english")

vectorizer = TfidfVectorizer(
    stop_words=stop_words,
    ngram_range=(1, 2),     # unigrams + bigrams
    max_features=100
)

# Fit TF-IDF
X = vectorizer.fit_transform(df["review text"])

# Get feature names
terms = vectorizer.get_feature_names_out()

# --------------------------------------
# 3. Extract Top Keywords per Bank
# --------------------------------------
banks = df["bank / app name"].unique()

bank_keywords = {}

for bank in banks:
    
    # Reviews for one bank
    bank_reviews = df[df["bank / app name"] == bank]["review text"]
    
    # Skip empty banks
    if len(bank_reviews) == 0:
        continue
    
    # TF-IDF for bank only
    X_bank = vectorizer.fit_transform(bank_reviews)
    
    # Average TF-IDF score
    scores = X_bank.mean(axis=0).A1
    
    # Get terms with scores
    term_scores = list(zip(vectorizer.get_feature_names_out(), scores))
    
    # Sort descending
    sorted_terms = sorted(term_scores, key=lambda x: x[1], reverse=True)
    
    # Top 15 keywords
    top_keywords = sorted_terms[:15]
    
    bank_keywords[bank] = top_keywords

# --------------------------------------
# 4. Display Results
# --------------------------------------
for bank, keywords in bank_keywords.items():
    
    print("\n" + "="*50)
    print(f"Bank: {bank}")
    print("="*50)
    
    for word, score in keywords:
        print(f"{word:<25} {score:.4f}")

# --------------------------------------
# 5. Save Keywords
# --------------------------------------
rows = []

for bank, keywords in bank_keywords.items():
    for word, score in keywords:
        rows.append([bank, word, score])

keywords_df = pd.DataFrame(rows, columns=["bank / app name", "keyword", "tfidf_score"])

keywords_df.to_csv("bank_keywords.csv", index=False)

print("\nThematic analysis completed.")


Bank: Commercial Bank of Ethiopia (CBE)
good                      0.1899
app                       0.0913
nice                      0.0599
best                      0.0587
ok                        0.0425
cbe                       0.0329
bank                      0.0268
working                   0.0242
use                       0.0218
update                    0.0213
like                      0.0206
excellent                 0.0187
fast                      0.0183
nice app                  0.0161
good app                  0.0160

Bank: Bank of Abyssinia (BOA)
good                      0.1616
app                       0.1192
best                      0.0507
nice                      0.0441
bank                      0.0321
boa                       0.0282
working                   0.0276
work                      0.0231
fast                      0.0228
banking                   0.0213
update                    0.0187
mobile                    0.0187
service                   0.0171
use 

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bewha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
